# Notebook 04 — Strategy Backtest

We now test whether the volatility risk premium identified in Notebook 03 
translates into real trading profits after costs and drawdowns.

**Strategies tested:**
1. Short ATM Straddle — sell call + put at same ATM strike
2. Short Strangle — sell OTM call + OTM put at fixed distance

**Data:** Synthetic NIFTY options chain generated via Black-Scholes.  
When real NSE bhavcopy files are available, swap `df = generate_synthetic_data()`  
with `df = load_bhavcopy_folder("../data/raw/")` and re-run.

**Backtest rules:**
- Entry: first trading day of each weekly expiry cycle
- Exit: expiry, stop-loss (1.5× entry premium), or target (50% capture)
- Costs: ₹50 per lot per leg
- Lot size: 75 (verify current NIFTY lot size from NSE)

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from option_pricing import bs_price
from backtester   import Backtester, NIFTY_LOT_SIZE
from metrics      import compute_metrics, equity_curve, drawdown_series, metrics_table
from plots        import plot_equity_curve

np.random.seed(42)
print("Imports OK. Lot size:", NIFTY_LOT_SIZE)

---
## 1. Generate synthetic NIFTY options data

We simulate 2 years of daily NIFTY options chains.

**Spot path:** Geometric Brownian Motion with:
- Annual drift μ = 12%
- Daily vol drawn from a realistic range (12%–25%) with persistence

**Option chain:** For each day and expiry, we price calls and puts at 
strikes from spot−1000 to spot+1000 in steps of 50.

In [ ]:
def generate_synthetic_data(
    start       = "2022-01-03",
    n_days      = 500,
    mu          = 0.12,
    base_sigma  = 0.15,
    lot_size    = NIFTY_LOT_SIZE,
):
    """
    Generate synthetic NIFTY F&O options chain data in the same format
    as NSE bhavcopy (after data_loader.py processing).
    
    Returns a DataFrame with columns:
        date, expiry_date, strike, option_type, close, spot, T
    """
    dates = pd.bdate_range(start, periods=n_days)
    
    # Simulate spot with stochastic vol (GARCH-like persistence)
    dt = 1/252
    spot_path = [19_000.0]
    vol_path  = [base_sigma]
    
    for _ in range(n_days - 1):
        # Vol mean-reverts with some noise
        prev_vol = vol_path[-1]
        vol_shock = np.random.normal(0, 0.015)
        new_vol   = np.clip(prev_vol * 0.95 + base_sigma * 0.05 + vol_shock, 0.08, 0.40)
        vol_path.append(new_vol)
        
        # Spot follows GBM
        ret = (mu - 0.5 * prev_vol**2) * dt + prev_vol * np.sqrt(dt) * np.random.normal()
        spot_path.append(spot_path[-1] * np.exp(ret))
    
    spots = pd.Series(spot_path, index=dates)
    vols  = pd.Series(vol_path,  index=dates)
    
    # Weekly expiries (every Thursday)
    all_thursdays = pd.date_range(start, periods=n_days + 60, freq="W-THU")
    expiries = [t for t in all_thursdays if t >= dates[0] and t <= dates[-1] + pd.Timedelta(days=30)]
    
    rows = []
    r = 0.065  # risk-free rate
    
    for date, spot, sigma_today in zip(dates, spots, vols):
        
        # Only use the next 3 expiries on any given day
        upcoming = [e for e in expiries if e >= date][:3]
        
        for expiry in upcoming:
            T = max((expiry - date).days / 365, 1/365)
            
            # Strikes: ATM ± 1000, step 50
            atm_strike = round(float(spot) / 50) * 50
            strikes = range(int(atm_strike) - 1000, int(atm_strike) + 1050, 50)
            
            for k in strikes:
                for ot, opt_str in [("CE", "call"), ("PE", "put")]:
                    try:
                        price = bs_price(float(spot), k, r, float(sigma_today), T, opt_str)
                        if price < 0.5:
                            continue   # skip near-zero deep OTM options
                        rows.append({
                            "date"        : date,
                            "expiry_date" : expiry,
                            "strike"      : float(k),
                            "option_type" : ot,
                            "close"       : round(price, 2),
                            "spot"        : round(float(spot), 2),
                            "T"           : round(T, 6),
                        })
                    except Exception:
                        pass
    
    df = pd.DataFrame(rows)
    print(f"Generated {len(df):,} option rows")
    print(f"Spot range: {spots.min():,.0f} – {spots.max():,.0f}")
    print(f"Vol  range: {vols.min():.1%} – {vols.max():.1%}")
    print(f"Expiries  : {len(expiries)}")
    print(f"Date range: {dates[0].date()} to {dates[-1].date()}")
    return df

df = generate_synthetic_data()
df.head(3)

In [ ]:
# Quick sanity check on the generated data
print("Columns:", df.columns.tolist())
print("Shape  :", df.shape)
print()
print("Sample ATM options on first trading day:")
first_day = df["date"].min()
first_exp = df[df["date"] == first_day]["expiry_date"].min()
sample = df[
    (df["date"] == first_day) &
    (df["expiry_date"] == first_exp)
].sort_values("strike")

# Show only strikes near spot
spot0 = sample["spot"].iloc[0]
near_atm = sample[(sample["strike"] >= spot0 - 200) & (sample["strike"] <= spot0 + 200)]
print(near_atm[["date","expiry_date","strike","option_type","close","T"]].to_string(index=False))

---
## 2. Run Short Straddle Backtest

**Entry:** Sell ATM call + ATM put on first day of each expiry cycle.  
**Exit:** Whichever comes first —
- Expiry (premium decays to intrinsic, often near zero)
- Stop-loss: current position cost > 1.5× entry premium
- Target: captured 50% of initial premium

In [ ]:
# Run short straddle
bt_straddle = Backtester(
    df,
    stop_loss_multiplier = 1.5,
    target_capture       = 0.50,
    otm_distance         = 150,
)
bt_straddle.run_short_straddle()
log_straddle = bt_straddle.trade_log()

print(f"Trades executed : {len(log_straddle)}")
print()
print(log_straddle[["entry_date","expiry_date","entry_total_premium",
                     "gross_pnl","net_pnl_rupees","exit_reason"]].to_string(index=False))

In [ ]:
# Performance metrics — short straddle
m_straddle = compute_metrics(log_straddle)

print("SHORT STRADDLE — Performance Metrics")
print("=" * 45)
for k, v in m_straddle.items():
    if k != "exit_reasons":
        print(f"  {k:<30}: {v}")

print()
print("Exit reasons:")
for reason, count in m_straddle["exit_reasons"].items():
    pct = count / m_straddle["num_trades"] * 100
    print(f"  {reason:<15}: {count} trades ({pct:.1f}%)")

In [ ]:
fig = plot_equity_curve(log_straddle, title="Short Straddle — Equity Curve")
plt.show()

---
## 3. Run Short Strangle Backtest

Same rules, but strikes are 150 points OTM instead of ATM.

**Tradeoff vs straddle:**
- Lower premium collected per trade
- Wider profit zone (more room before you lose)
- Lower win rate but smaller average loss when stop-loss hits

In [ ]:
bt_strangle = Backtester(
    df,
    stop_loss_multiplier = 1.5,
    target_capture       = 0.50,
    otm_distance         = 150,
)
bt_strangle.run_short_strangle()
log_strangle = bt_strangle.trade_log()

print(f"Trades executed : {len(log_strangle)}")
print()
m_strangle = compute_metrics(log_strangle)

print("SHORT STRANGLE — Performance Metrics")
print("=" * 45)
for k, v in m_strangle.items():
    if k != "exit_reasons":
        print(f"  {k:<30}: {v}")

In [ ]:
fig = plot_equity_curve(log_strangle, title="Short Strangle — Equity Curve")
plt.show()

---
## 4. Side-by-side comparison

In [ ]:
# Equity curves on same chart
eq_straddle = equity_curve(log_straddle)
eq_strangle = equity_curve(log_strangle)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(eq_straddle.index, eq_straddle, color="steelblue",  lw=2.5, label="Short Straddle")
ax.plot(eq_strangle.index, eq_strangle, color="darkorange", lw=2.5, label="Short Strangle", ls="--")
ax.axhline(0, color="gray", lw=1)
ax.set_title("Short Straddle vs Short Strangle — Cumulative P&L", fontsize=13, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Cumulative P&L (Rs)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"Rs{x:,.0f}"))
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Summary comparison table
metrics_keys = ["total_return","avg_trade_pnl","win_rate_pct","num_trades",
                "max_drawdown","sharpe_ratio","profit_factor","best_trade","worst_trade"]

comparison = pd.DataFrame({
    "Short Straddle": {k: m_straddle.get(k) for k in metrics_keys},
    "Short Strangle": {k: m_strangle.get(k) for k in metrics_keys},
})
print("Strategy Comparison")
print(comparison.to_string())

---
## 5. Sensitivity analysis — stop-loss level

How sensitive are results to the stop-loss multiplier?
We test 1.0×, 1.5×, 2.0×, and 3.0× for the straddle.

In [ ]:
sl_levels = [1.0, 1.5, 2.0, 3.0]
results = []

for sl in sl_levels:
    bt = Backtester(df, stop_loss_multiplier=sl, target_capture=0.50)
    bt.run_short_straddle()
    log = bt.trade_log()
    if log.empty:
        continue
    m = compute_metrics(log)
    results.append({
        "SL multiplier"  : sl,
        "Total return Rs": m["total_return"],
        "Win rate %"     : m["win_rate_pct"],
        "Max drawdown Rs": m["max_drawdown"],
        "Sharpe"         : m["sharpe_ratio"],
        "Profit factor"  : m["profit_factor"],
        "Trades"         : m["num_trades"],
    })

sens_df = pd.DataFrame(results)
print("Stop-loss sensitivity — Short Straddle")
print(sens_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, col, color in [
    (axes[0], "Total return Rs", "steelblue"),
    (axes[1], "Max drawdown Rs", "tomato"),
    (axes[2], "Sharpe",          "green"),
]:
    ax.bar(sens_df["SL multiplier"].astype(str), sens_df[col], color=color, alpha=0.8)
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("Stop-loss multiplier")
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Sensitivity to Stop-Loss Level — Short Straddle", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

---
## 6. VIX regime filter

From Notebook 03, high-VIX periods are riskier.
Let us test whether filtering out entries when VIX percentile > 80 improves the strategy.

In [ ]:
import os

# Load VIX percentile if available, else skip filter
vix_pct_path = "../data/processed/vix_percentile.csv"

if os.path.exists(vix_pct_path):
    vix_pct = pd.read_csv(vix_pct_path, index_col=0, parse_dates=True).squeeze()
    
    # Filter trades where VIX percentile at entry was < 80th
    log_filtered = log_straddle.copy()
    log_filtered["entry_date"] = pd.to_datetime(log_filtered["entry_date"])
    log_filtered = log_filtered.join(
        vix_pct.rename("vix_pct"), on="entry_date", how="left"
    )
    log_filtered = log_filtered[log_filtered["vix_pct"] < 0.80]

    print(f"Unfiltered trades : {len(log_straddle)}")
    print(f"Filtered trades   : {len(log_filtered)}  (removed high-VIX entries)")
    
    if not log_filtered.empty:
        m_filtered = compute_metrics(log_filtered)
        print()
        print(f"  Total return  : Rs{m_filtered['total_return']:,.0f}")
        print(f"  Win rate      : {m_filtered['win_rate_pct']:.1f}%")
        print(f"  Max drawdown  : Rs{m_filtered['max_drawdown']:,.0f}")
        print(f"  Sharpe        : {m_filtered['sharpe_ratio']}")
else:
    print("VIX percentile file not found — run Notebook 03 first to generate it.")
    print("Skipping regime filter.")

---
## 7. Save trade logs

Save both trade logs for the dashboard and final report.

In [ ]:
log_straddle.to_csv("../data/processed/backtest_straddle.csv", index=False)
log_strangle.to_csv("../data/processed/backtest_strangle.csv", index=False)
print("Saved backtest_straddle.csv and backtest_strangle.csv")
print()
print("These will be picked up automatically by the Streamlit dashboard.")

---
## Key takeaways

1. **Short straddle wins** by collecting premium that decays as expiry approaches.
2. **Stop-loss is essential** — an unhedged short straddle during a large move can be catastrophic.
3. **Target exits** improve Sharpe by locking in profits early and redeploying capital.
4. **Strangle vs straddle** — strangle has lower premium but more breathing room. 
   Neither dominates universally — depends on the vol regime.
5. **Results are on synthetic data.** Validate on real NSE bhavcopy data before drawing conclusions.

**Next:** Notebook 05 — Delta Hedging simulation.